In [0]:
silver = spark.read.table("my_ecommerce_store.silver.silver_orders")

from pyspark.sql.functions import expr, col

gold_base = silver.withColumn(
    "total_amount",
    col("price") + col("payment_value")
)

In [0]:
from pyspark.sql.functions import sum, count, to_date

daily_sales = gold_base.groupBy(
    to_date("order_purchase_timestamp").alias("order_date")
).agg(
    sum("total_amount").alias("total_sales"),
    count("order_id").alias("total_orders")
)

In [0]:
daily_sales.show(2)
gold_base.show(2)

+----------+-----------------+------------+
|order_date|      total_sales|total_orders|
+----------+-----------------+------------+
|2018-08-10|78033.23999999999|         286|
|2017-08-11|         50334.99|         176|
+----------+-----------------+------------+
only showing top 2 rows
+--------------------+--------------------+------------+------------------------+---------------------+-----+-------------+------------+
|            order_id|         customer_id|order_status|order_purchase_timestamp|product_category_name|price|payment_value|total_amount|
+--------------------+--------------------+------------+------------------------+---------------------+-----+-------------+------------+
|e481f51cbdc54678b...|9ef432eb625129730...|   delivered|     2017-10-02 10:56:33| utilidades_domest...|29.99|        18.59|       48.58|
|e481f51cbdc54678b...|9ef432eb625129730...|   delivered|     2017-10-02 10:56:33| utilidades_domest...|29.99|          2.0|       31.99|
+--------------------+-----

In [0]:
daily_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("my_ecommerce_store.gold.daily_sales")

In [0]:
from pyspark.sql.functions import sum, count, to_date

daily_sales = gold_base.groupBy(
    to_date("order_purchase_timestamp").alias("order_date")
).agg(
    sum("total_amount").alias("total_sales"),
    count("order_id").alias("total_orders")
)

In [0]:
from pyspark.sql.functions import sum

category_sales = gold_base.groupBy(
    "product_category_name"
).agg(
    sum("total_amount").alias("total_revenue"),
    count("order_id").alias("total_orders")
).orderBy("total_revenue", ascending=False)

In [0]:
category_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("my_ecommerce_store.gold.category_sales")

In [0]:
from pyspark.sql.functions import sum, count, to_date

daily_sales = gold_base.groupBy(
    to_date("order_purchase_timestamp").alias("order_date")
).agg(
    sum("total_amount").alias("total_sales"),
    count("order_id").alias("total_orders")
)

In [0]:
daily_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("my_ecommerce_store.gold.daily_sales")

In [0]:
spark.conf.set(
    "fs.azure.account.key.mystoarageadls001.dfs.core.windows.net",
    "ntKntWtYQvzy3nIpuFr8RmrskGySDFmJEFA688uXb7kQn1ZphKiPyKcB0y0yHUD+fLD561/GyXFU+AStD6192Q=="
)

daily_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .save("abfss://mystoragevault@mystoarageadls001.dfs.core.windows.net/gold/daily_sales/")

gold_base.write.format("delta") \
    .mode("overwrite") \
    .save("abfss://mystoragevault@mystoarageadls001.dfs.core.windows.net/gold/base/")

category_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save("abfss://mystoragevault@mystoarageadls001.dfs.core.windows.net/gold/category_sales/")